# XGBoost CatBoost Baseline - LB 0.688
### XGBoost Classifier + XGBoost & CatBoost Regressor Ensemble Method 
### Baseline code of this notebook and some functions are written with reference to a public notebook. 
### Thank you to everyone who shared

# Pip Install Libraries for Metric
Since internet must be turned off for submission, we pip install from my other notebook [here][1] where I downloaded the WHL files.

[1]: https://www.kaggle.com/code/cdeotte/pip-install-lifelines

In [ ]:
!pip install /kaggle/input/pip-install-lifelines/autograd-1.7.0-py3-none-any.whl
!pip install /kaggle/input/pip-install-lifelines/autograd-gamma-0.5.0.tar.gz
!pip install /kaggle/input/pip-install-lifelines/interface_meta-1.3.0-py3-none-any.whl
!pip install /kaggle/input/pip-install-lifelines/formulaic-1.0.2-py3-none-any.whl
!pip install /kaggle/input/pip-install-lifelines/lifelines-0.30.0-py3-none-any.whl

# Load Train and Test

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 500)

test = pd.read_csv("/kaggle/input/equity-post-HCT-survival-predictions/test.csv")
print("Test shape:", test.shape )

train = pd.read_csv("/kaggle/input/equity-post-HCT-survival-predictions/train.csv")
print("Train shape:",train.shape)
train.head()

# EDA on Train Targets
There are two train targets `efs` and `efs_time`. When `efs==1` we know patient **did not survive** and we know time of death is `efs_time`. When `efs==0` we **do not know** if patient survived or not, but we do know that patient survived at least as long as `efs_time`.

In [ ]:
plt.hist(train.loc[train.efs==1,"efs_time"],bins=100,label="efs=1, Did Not Survive")
plt.hist(train.loc[train.efs==0,"efs_time"],bins=100,label="efs=0, Maybe Survived")
plt.xlabel("Time of Observation, efs_time")
plt.ylabel("Density")
plt.title("Times of Observation. Either time to death, or time observed alive.")
plt.legend()
plt.show()

# Transform Two Train Targets into One Target!
Both targets `efs` and `efs_time` provide useful information. We will tranform these two targets into a single target to train our model with.

In [ ]:
from lifelines import KaplanMeierFitter

def transform_survival_probability(df, time_col='efs_time', event_col='efs'):
    """
    Transform using survival probability estimates
    """
    kmf = KaplanMeierFitter()
    kmf.fit(df[time_col], df[event_col])
    
    # Get survival probabilities at each time point
    y = kmf.survival_function_at_times(df[time_col]).values
    
    # Adjust for censoring
    # censored_mask = df[event_col] == 0
    #y[censored_mask] = y[censored_mask] * 1.2  # Increase survival prob for censored
    
    return y

train["y"] = transform_survival_probability(train, time_col='efs_time', event_col='efs')

In [ ]:
plt.hist(train.loc[train.efs==1,"y"],bins=100,label="efs=1, Did Not Survive")
plt.hist(train.loc[train.efs==0,"y"],bins=100,label="efs=0, Maybe Survived")
plt.xlabel("Transformed Target y")
plt.ylabel("Density")
plt.title("Transformed Target y using both efs and efs_time.")
plt.legend()
plt.show()

# Features
### 

In [ ]:
RMV = ["ID","efs","efs_time","y"]
FEATURES = [c for c in train.columns if not c in RMV]
print(f"There are {len(FEATURES)} FEATURES: {FEATURES}")

In [ ]:
hct_ci_mapping = {
    "arrhythmia": {"No": 0, "Not done": 0, "Yes": 1},  
    "cardiac": {"No": 0, "Not done": 0, "Yes": 1}, 
    "diabetes": {"No": 0, "Not done": 0, "Yes": 1},  
    "hepatic_mild": {"No": 0, "Not done": 0, "Yes": 1},
    "hepatic_severe": {"No": 0, "Not done": 0, "Yes": 3},
    "psych_disturb": {"No": 0, "Not done": 0, "Yes": 1}, 
    "obesity": {"No": 0, "Not done": 0, "Yes": 1}, 
    "rheum_issue": {"No": 0, "Not done": 0, "Yes": 2},
    "peptic_ulcer": {"No": 0, "Not done": 0, "Yes": 2},  
    "renal_issue": {"No": 0, "Not done": 0, "Yes": 2}, 
    "prior_tumor": {"No": 0, "Not done": 0, "Yes": 3}, 
    "pulm_moderate": {"No": 0, "Not done": 0, "Yes": 2}, 
    "pulm_severe": {"No": 0, "Not done": 0, "Yes": 3},  
}
def calculate_hct_ci_score(row, mapping):
        """
        This function calculates the hct_ci score
    
        Args:
            row (pd.Series): Patient Clinical Data
            mapping (dict): HCT-CI score mapping
    
        Returns:
            int: HCT-CI score
        """
    
        score = 0
    
        if "hepatic_severe" in row and row["hepatic_severe"] == "Yes":
            score += mapping["hepatic_severe"]["Yes"]
        elif "hepatic_mild" in row and row["hepatic_mild"] == "Yes":
            score += mapping["hepatic_mild"]["Yes"]
        if "pulm_moderate" in row and row["pulm_moderate"] == "Yes":
            score += mapping["pulm_moderate"]["Yes"]
        elif "pulm_severe" in row and row["pulm_severe"] == "Yes":
            score += mapping["pulm_severe"]["Yes"]
    
        # Other Conditions
        for condition, mapping_values in mapping.items():
            if condition not in ["hepatic_mild", "hepatic_severe","pulm_moderate", "pulm_severe"] and condition in row:
                score += mapping_values.get(row[condition], 0)
    
        return score

In [ ]:
# cat2num function is used for mapping some of the Categorical Values into Numerical Values

def cat2num(df):
    df['conditioning_intensity'] = df['conditioning_intensity'].map({
    'NMA': 1, 
    'RIC': 2,
    'MAC': 3,
    'TBD': None,
    'No drugs reported': None,
    'N/A, F(pre-TED) not submitted': None})
    
    df['tbi_status'] = df['tbi_status'].map({
    'No TBI': 0, 
    'TBI +- Other, <=cGy': 1,
    'TBI +- Other, -cGy, fractionated': 2,
    'TBI + Cy +- Other': 3,
    'TBI +- Other, -cGy, single': 4,
    'TBI +- Other, >cGy': 5,
    'TBI +- Other, unknown dose': None})
    
    df['dri_score'] = df['dri_score'].map({
    'Low': 1, 
    'Intermediate': 2,
    'Intermediate - TED AML case <missing cytogenetics': 3,
    'High': 4,
    'High - TED AML case <missing cytogenetics': 5,
    'Very High': 6,
    'N/A - pediatric': -3,
    'N/A - non-malignant indication': -1,
    'TBD cytogenetics': -2,
    'N/A - disease not classifiable': -4,
    'Missing disease status': 0})
    
    df['cyto_score'] = df['cyto_score'].map({
    'Poor': 4,
    'Normal': 3,
    'Intermediate': 2,
    'Favorable': 1,
    'TBD': -1,
    'Other': -2,
    'Not tested': None})
    
    df['cyto_score_detail'] = df['cyto_score_detail'].map({
    'Poor': 3, 
    'Intermediate': 2,
    'Favorable': 1,
    'TBD': -1,
    'Not tested': None})
    
    return df

In [ ]:
def fill_hla_combined_low(row):
    if np.isnan(row['hla_combined_low']): 
        components = [
            row['hla_match_drb1_low'], row['hla_match_dqb1_low'], 
            row['hla_match_a_low'], row['hla_match_b_low'], row['hla_match_c_low']
        ]
        if all([not np.isnan(x) for x in components]):
            return sum(components)
        else:
            if not np.isnan(row['hla_low_res_8']) and not np.isnan(row['hla_match_dqb1_low']):
                return row['hla_low_res_8'] + row['hla_match_dqb1_low']
            elif not np.isnan(row['hla_low_res_6']): 
                components_6 = [
                    row['hla_match_dqb1_low'], row['hla_match_c_low']
                ]
                if all([not np.isnan(x) for x in components_6]):
                    return row['hla_low_res_6'] + sum(components_6)
                else: 
                    return sum([x for x in components if not np.isnan(x)])
    return row['hla_combined_low'] 

In [ ]:
def add_features(df):
    df["hct_ci_score"] = df.apply(lambda row: calculate_hct_ci_score(row, hct_ci_mapping), axis=1)
    df['donor_recipient_age_diff'] = abs(df['donor_age'] - df['age_at_hct'])
    df = cat2num(df)
    df['hla_combined_low'] = df['hla_low_res_10']
    df['hla_combined_low'] = df.apply(fill_hla_combined_low, axis=1)
    df['hla_match_ratio'] = (df['hla_high_res_8'] + df['hla_low_res_8']) / 16
    df['years_since_2000'] = df['year_hct'] - 2000
    df['null_count'] = df.isnull().sum(axis=1)
    df['ci_score_danger'] = df['hct_ci_score'].apply(lambda x: 2 if x >= 3 else 1 if x >= 1 else 0)
    return df

train = add_features(train)
test = add_features(test)

In [ ]:
# train["hct_ci_score"] = train.apply(lambda row: calculate_hct_ci_score(row, hct_ci_mapping), axis=1)
# test["hct_ci_score"] = test.apply(lambda row: calculate_hct_ci_score(row, hct_ci_mapping), axis=1)
# train = cat2num(train)
# test = cat2num(test)
# train = recalculate_hla_sums(train)
# test = recalculate_hla_sums(test)

In [ ]:
CATS = []
for c in FEATURES:
    if train[c].dtype=="object":
        CATS.append(c)
        train[c] = train[c].fillna("NAN")
        test[c] = test[c].fillna("NAN")
print(f"In these features, there are {len(CATS)} CATEGORICAL FEATURES: {CATS}")

In [ ]:
# import pandas as pd
# import matplotlib.pyplot as plt

# plt.hist(train['null_count'], bins=100, edgecolor='black')  # bins를 조절하여 구간 개수 변경 가능
# plt.xlabel('null count')
# plt.ylabel('Frequency')
# plt.title('null count histogram')
# plt.show()

In [ ]:
combined = pd.concat([train,test],axis=0,ignore_index=True)
#print("Combined data shape:", combined.shape )

# LABEL ENCODE CATEGORICAL FEATURES
print("We LABEL ENCODE the CATEGORICAL FEATURES: ",end="")
for c in FEATURES:

    # LABEL ENCODE CATEGORICAL AND CONVERT TO INT32 CATEGORY
    if c in CATS:
        print(f"{c}, ",end="")
        combined[c],_ = combined[c].factorize()
        combined[c] -= combined[c].min()
        combined[c] = combined[c].astype("int32")
        combined[c] = combined[c].astype("category")
        
    # REDUCE PRECISION OF NUMERICAL TO 32BIT TO SAVE MEMORY
    else:
        if combined[c].dtype=="float64":
            combined[c] = combined[c].astype("float32")
        if combined[c].dtype=="int64":
            combined[c] = combined[c].astype("int32")

# for c in cat2num:
#     combined[c] = combined[c].astype("int32")

train = combined.iloc[:len(train)].copy()
test = combined.iloc[len(train):].reset_index(drop=True).copy()

In [ ]:
FEATURES += ["hct_ci_score", 'donor_recipient_age_diff', "hla_combined_low", "hla_match_ratio", 
             "years_since_2000", "null_count","ci_score_danger"]

In [ ]:
train.head()

# XGBoost Classifier

In [ ]:
from sklearn.model_selection import KFold
from xgboost import XGBRegressor, XGBClassifier
import xgboost
print("Using XGBoost version",xgboost.__version__)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier

FOLDS = 5
kf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=42)

oof_xgb = np.zeros(len(train))
pred_efs = np.zeros(len(test))

for i, (train_index, test_index) in enumerate(kf.split(train, train["efs"])):

    print("#"*25)
    print(f"### Fold {i+1}")
    print("#"*25)
    
    x_train = train.loc[train_index, FEATURES].copy()
    y_train = train.loc[train_index, "efs"]
    x_valid = train.loc[test_index, FEATURES].copy()
    y_valid = train.loc[test_index, "efs"]
    x_test = test[FEATURES].copy()

    model_xgb = XGBClassifier(
        device="cuda",
        max_depth=3,  
        colsample_bytree=0.7129400756425178, 
        subsample=0.8185881823156917, 
        n_estimators=20_000, 
        learning_rate=0.04425768131771064,  
        eval_metric="auc", 
        early_stopping_rounds=50, 
        objective='binary:logistic',
        scale_pos_weight=1.5379160847615545,  
        min_child_weight=4,
        enable_categorical=True,
        gamma=3.1330719334577584
    )
    model_xgb.fit(
        x_train, y_train,
        eval_set=[(x_valid, y_valid)],  
        verbose=100
    )

    # INFER OOF (Probabilities -> Binary)
    oof_xgb[test_index] = (model_xgb.predict_proba(x_valid)[:, 1] > 0.5).astype(int)
    # INFER TEST (Probabilities -> Average Probs)
    pred_efs += model_xgb.predict_proba(x_test)[:, 1]

# COMPUTE AVERAGE TEST PREDS
pred_efs = (pred_efs / FOLDS > 0.5).astype(int)

# EVALUATE PERFORMANCE
accuracy = accuracy_score(train["efs"], oof_xgb)
f1 = f1_score(train["efs"], oof_xgb)
roc_auc = roc_auc_score(train["efs"], oof_xgb)

print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"ROC AUC Score: {roc_auc:.4f}")


In [ ]:
bin_pred = oof_xgb
bin_pred

In [ ]:
pred_classifier = pred_efs
pred_classifier

# XGBoost Regressor

In [ ]:
%%time
FOLDS = 5
kf = KFold(n_splits=FOLDS, shuffle=True, random_state=42)
    
oof_xgb = np.zeros(len(train))
pred_xgb = np.zeros(len(test))

for i, (train_index, test_index) in enumerate(kf.split(train)):

    print("#"*25)
    print(f"### Fold {i+1}")
    print("#"*25)
    
    x_train = train.loc[train_index,FEATURES].copy()
    y_train = train.loc[train_index,"y"]
    x_valid = train.loc[test_index,FEATURES].copy()
    y_valid = train.loc[test_index,"y"]
    x_test = test[FEATURES].copy()
    
    model_xgb = XGBRegressor(
        device="cpu",
        max_depth=5,  
        colsample_bytree=0.4309907360736148, 
        subsample=0.6727848987288046, 
        n_estimators=10_000,  
        learning_rate=0.03509792076095853, 
        eval_metric="mae",
        early_stopping_rounds=25,
        objective='reg:logistic',
        enable_categorical=True,
        min_child_weight=10,
        reg_alpha= 2.950200470036872, 
        reg_lambda= 1.484334590329492,
        gamma = 0.008314053362236895
    )
    model_xgb.fit(
        x_train, y_train,
        eval_set=[(x_valid, y_valid)],  
        verbose=100 
    )

    # INFER OOF
    oof_xgb[test_index] = model_xgb.predict(x_valid)
    # INFER TEST
    pred_xgb += model_xgb.predict(x_test)

# COMPUTE AVERAGE TEST PREDS
pred_xgb /= FOLDS

In [ ]:
oof_xgb

In [ ]:
pred_xgb

In [ ]:
bin_pred_np = np.array(bin_pred)
oof_xgb_np = np.array(oof_xgb)

combined_array = np.column_stack((bin_pred_np, oof_xgb_np))

print(combined_array)  # (2, 28800)

In [ ]:
data_0 = combined_array[combined_array[:, 0] == 0]
data_1 = combined_array[combined_array[:, 0] == 1]


plt.figure(figsize=(10, 6))
plt.hist(data_0[:, 1], bins=100, color='blue', alpha=0.6, label='bin_pred = 0')
plt.hist(data_1[:, 1], bins=100, color='red', alpha=0.6, label='bin_pred = 1')

plt.xlabel('oof_xgb values (Second Column)')
plt.ylabel('Density')
plt.title('Histogram of oof_xgb values by bin_pred')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
combined_array[combined_array[:, 0] == 1, 1] += 0.1

In [ ]:
data_0 = combined_array[combined_array[:, 0] == 0]
data_1 = combined_array[combined_array[:, 0] == 1]

plt.figure(figsize=(10, 6))

plt.hist(data_0[:, 1], bins=100, color='blue', alpha=0.6, label='bin_pred = 0')

plt.hist(data_1[:, 1], bins=100, color='red', alpha=0.6, label='bin_pred = 1')

plt.xlabel('oof_xgb values (Second Column)')
plt.ylabel('Density')
plt.title('Histogram of oof_xgb values by bin_pred')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
y_true = train[["ID","efs","efs_time","race_group"]].copy()
y_pred = train[["ID"]].copy()
y_pred["prediction"] = combined_array[:, 1]

print(y_pred)

In [ ]:
from metric import score

m = score(y_true.copy(), y_pred.copy(), "ID")
print(f"\nOverall CV for XGBoost =",m)

In [ ]:
feature_importance = model_xgb.feature_importances_
importance_df = pd.DataFrame({
    "Feature": FEATURES,  # Replace FEATURES with your list of feature names
    "Importance": feature_importance
}).sort_values(by="Importance", ascending=False)
plt.figure(figsize=(10, 15))
plt.barh(importance_df["Feature"], importance_df["Importance"])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("XGBoost Feature Importance")
plt.gca().invert_yaxis()  # Flip features for better readability
plt.show()

# CatBoost
We train CatBoost model with CV 0.665

In [ ]:
from catboost import CatBoostRegressor, CatBoostClassifier
import catboost
print("Using CatBoost version",catboost.__version__)

In [ ]:
def optimize_catboost(train, FEATURES, CATS, n_trials=30):
    def objective(trial):
        params = {
            'depth': trial.suggest_int('depth', 4, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 15),
            'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.6, 1.0),
            'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 30),
            'grow_policy': trial.suggest_categorical('grow_policy', ['SymmetricTree', 'Depthwise', 'Lossguide']),
            'bootstrap_type': trial.suggest_categorical('bootstrap_type', ['Bayesian', 'Bernoulli', 'MVS']),
            'iterations': trial.suggest_int('iterations', 800, 2000),
        }

        if params['bootstrap_type'] == 'Bayesian':
            params['bagging_temperature'] = trial.suggest_float('bagging_temperature', 0.1, 10.0)

        FOLDS = 5
        kf = KFold(n_splits=FOLDS, shuffle=True, random_state=42)
        mae_scores = []

        for train_idx, valid_idx in kf.split(train):
            X_train = train.iloc[train_idx][FEATURES]
            y_train = train.iloc[train_idx]['y']
            X_valid = train.iloc[valid_idx][FEATURES]
            y_valid = train.iloc[valid_idx]['y']

            model = CatBoostRegressor(
                **params,
                cat_features=CATS,
                task_type="CPU",  # GPU 사용 시 'GPU'로 변경
                eval_metric='MAE',
                early_stopping_rounds=100,
                random_seed=42,
                verbose=0
            )

            model.fit(
                X_train, y_train,
                eval_set=(X_valid, y_valid),
                use_best_model=True
            )

            preds = model.predict(X_valid)
            mae = mean_absolute_error(y_valid, preds)
            mae_scores.append(mae)

        return np.mean(mae_scores)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials)
    return study.best_params

# best_params = optimize_catboost(train, FEATURES, CATS, n_trials=30)
# print(best_params)

In [ ]:
%%time
FOLDS = 5
kf = KFold(n_splits=FOLDS, shuffle=True, random_state=42)

oof_cat = np.zeros(len(train))
pred_cat = np.zeros(len(test))
# cat_params = {
#         "iterations": 1000,
#         "depth": 6,
#         "learning_rate": 0.2,
#         "l2_leaf_reg": 5,
#         "random_seed": 42,
#         "task_type": "CPU",
#         "verbose": 100
#     }
cat_params = {
    'depth': 6, 
    'learning_rate': 0.04699005545173896, 
    'l2_leaf_reg': 6.853082507365295, 
    'colsample_bylevel': 0.9312642681213008, 
    'min_data_in_leaf': 14, 
    'grow_policy': 'Depthwise', 
    'bootstrap_type': 'Bernoulli', 
    'iterations': 1727
}
for i, (train_index, test_index) in enumerate(kf.split(train)):

    print("#"*25)
    print(f"### Fold {i+1}")
    print("#"*25)

    x_train = train.loc[train_index,FEATURES].copy()
    y_train = train.loc[train_index,"y"]
    x_valid = train.loc[test_index,FEATURES].copy()
    y_valid = train.loc[test_index,"y"]
    x_test = test[FEATURES].copy()

    model_cat = CatBoostRegressor(
        #**best_params,
        **cat_params,
        cat_features=CATS,
        task_type="CPU",  
        eval_metric='MAE',
        early_stopping_rounds=100,
        random_seed=42,
        verbose=100
    )
    
    model_cat.fit(
        x_train,
        y_train,
        eval_set=(x_valid, y_valid),
    )

    # INFER OOF
    oof_cat[test_index] = model_cat.predict(x_valid)
    # INFER TEST
    pred_cat += model_cat.predict(x_test)

# COMPUTE AVERAGE TEST PREDS
pred_cat /= FOLDS

In [ ]:
bin_pred_np = np.array(bin_pred)
oof_cat_np = np.array(oof_cat)

combined_array_cat = np.column_stack((bin_pred_np, oof_cat_np))

print(combined_array)  # (2, 28800)

In [ ]:
data_0 = combined_array_cat[combined_array_cat[:, 0] == 0]
data_1 = combined_array_cat[combined_array_cat[:, 0] == 1]

plt.figure(figsize=(10, 6))

plt.hist(data_0[:, 1], bins=100, color='blue', alpha=0.6, label='bin_pred = 0')

plt.hist(data_1[:, 1], bins=100, color='red', alpha=0.6, label='bin_pred = 1')

# 레이블과 제목 추가
plt.xlabel('oof_cat values (Second Column)')
plt.ylabel('Density')
plt.title('Histogram of oof_cat values by bin_pred')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
combined_array_cat[combined_array_cat[:, 0] == 1, 1] += 0.1


In [ ]:
data_0 = combined_array_cat[combined_array_cat[:, 0] == 0]
data_1 = combined_array_cat[combined_array_cat[:, 0] == 1]

plt.figure(figsize=(10, 6))

plt.hist(data_0[:, 1], bins=100, color='blue', alpha=0.6, label='bin_pred = 0')

plt.hist(data_1[:, 1], bins=100, color='red', alpha=0.6, label='bin_pred = 1')

plt.xlabel('oof_cat values (Second Column)')
plt.ylabel('Density')
plt.title('Histogram of oof_cat values by bin_pred')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
y_true = train[["ID","efs","efs_time","race_group"]].copy()
y_pred = train[["ID"]].copy()
y_pred["prediction"] = combined_array_cat[:, 1]

print(y_pred)

In [ ]:
m = score(y_true.copy(), y_pred.copy(), "ID")
print(f"\nOverall CV for CatBoost =",m)


# CatBoost Feature Importance

In [ ]:
feature_importance = model_cat.get_feature_importance()
importance_df = pd.DataFrame({
    "Feature": FEATURES, 
    "Importance": feature_importance
}).sort_values(by="Importance", ascending=False)
plt.figure(figsize=(10, 15))
plt.barh(importance_df["Feature"], importance_df["Importance"])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("CatBoost Feature Importance")
plt.gca().invert_yaxis()  # Flip features for better readability
plt.show()

# Ensemble CAT and XGB
We ensemble our XGBoost and CatBoost to achieve CV 0.668!

In [ ]:
y_true = train[["ID","efs","efs_time","race_group"]].copy()
y_pred = train[["ID"]].copy()
y_pred["prediction"] = combined_array[:, 1] + combined_array_cat[:, 1]
m = score(y_true.copy(), y_pred.copy(), "ID")
print(f"\nOverall CV for Ensemble =",m)

# Create Submission CSV

In [ ]:
prediction = pred_xgb + pred_cat

pred_classifier_np = np.array(pred_classifier)
prediction_np = np.array(prediction)

combined_pred = np.column_stack((pred_classifier_np, prediction_np))
print(combined_pred)  # (2, 3)

combined_pred[combined_pred[:, 0] == 1, 1] += 0.1
print(combined_pred)  # (2, 3)


sub = pd.read_csv("/kaggle/input/equity-post-HCT-survival-predictions/sample_submission.csv")
sub.prediction = combined_pred[:, 1]

print(sub)


In [ ]:
sub.to_csv("submission.csv",index=False)
print("Sub shape:",sub.shape)
sub.head()